# 06 – Valutazione della Pipeline Classica

| Sezione | Cosa misuriamo |
|---------|---------------|
| 1. Metriche interne | Silhouette, Davies-Bouldin, Calinski-Harabasz |
| 2. Stabilita bootstrap | Robustezza: stesse rotte anomale su sottocampioni? |
| 3. Sensitivity | Impatto delle soglie sui risultati |
| 4. Feature importance | Permutation importance (IsolationForest) |
| 5. Validazione qualitativa | Le anomalie hanno profilo operativo coerente? |
| **6. SHAP Explainability** | **Quale feature spiega ogni singola anomalia?** |
| 7. Scorecard finale | Riepilogo per confronto con Multi-Agent |

**Input**: `anomaly_results.csv` · `features_with_baseline.csv` · `final_report.csv`

In [1]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, GradientBoostingClassifier
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

ROOT     = Path("..")
PROC_DIR = ROOT / "data" / "processed"
print("Librerie caricate")

Librerie caricate


## 1. Caricamento dati

In [2]:
ar  = pd.read_csv(PROC_DIR / "anomaly_results.csv")
fb  = pd.read_csv(PROC_DIR / "features_with_baseline.csv")
rep = pd.read_csv(PROC_DIR / "final_report.csv")
bs  = json.loads((PROC_DIR / "baseline_stats.json").read_text())["stats"]

ANOMALY_FEATURES = [
    "tot_allarmi_log","pct_interpol","pct_sdi","pct_nsis",
    "tasso_chiusura","tasso_rilevanza","tasso_allarme_medio",
    "tasso_inv_medio","score_rischio_esiti","tasso_respinti","tasso_fermati",
]

X_raw    = fb[ANOMALY_FEATURES].fillna(0).values
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_scaled_df = pd.DataFrame(X_scaled, columns=ANOMALY_FEATURES, index=fb.index)

labels_ml   = ar["anomaly_label"].map({"ALTA":1,"MEDIA":1,"NORMALE":0}).values
labels_risk = rep["risk_level"].map({"CRITICO":1,"ALTO":1,"MEDIO":1,"BASSO":0}).values

print(f"Feature matrix X : {X_scaled.shape}")
print(f"Anomalie ML (ALTA+MEDIA): {labels_ml.sum()} ({100*labels_ml.mean():.1f}%)")
print(f"Rotte in allerta (05)   : {labels_risk.sum()} ({100*labels_risk.mean():.1f}%)")

Feature matrix X : (567, 11)
Anomalie ML (ALTA+MEDIA): 57 (10.1%)
Rotte in allerta (05)   : 57 (10.1%)


## 2. Metriche Interne di Clustering

- **Silhouette** [-1,1]: piu alto = cluster piu separati (>0.3 accettabile, >0.5 buono)
- **Davies-Bouldin** [>=0]: piu basso = cluster piu compatti (<1.0 buono)
- **Calinski-Harabasz**: piu alto = cluster piu densi e separati

> Nota: con 4.8% anomalie vs 95.2% normali, Silhouette e strutturalmente penalizzato  
> dall'imbalance di classe — e atteso che sia basso.

In [3]:
sil = silhouette_score(X_scaled, labels_ml)
db  = davies_bouldin_score(X_scaled, labels_ml)
ch  = calinski_harabasz_score(X_scaled, labels_ml)

sil_r = silhouette_score(X_scaled, labels_risk)
db_r  = davies_bouldin_score(X_scaled, labels_risk)
ch_r  = calinski_harabasz_score(X_scaled, labels_risk)

h1 = "Metrica"
h2 = "Label ML (04)"
h3 = "Label Risk (05)"
print("=== METRICHE INTERNE ===")
print(f"{h1:<25} {h2:<20} {h3}")
print("-" * 65)
for name, v1, v2 in [("Silhouette Score", sil, sil_r),
                      ("Davies-Bouldin",   db,  db_r),
                      ("Calinski-Harabasz",ch,  ch_r)]:
    print(f"{name:<25} {v1:<20.4f} {v2:.4f}")
print()
for val, label in [(sil,"ML"), (sil_r,"Risk")]:
    if val >= 0.50:   interp = "Buona separazione"
    elif val >= 0.25: interp = "Separazione accettabile"
    else:             interp = "Sovrapposizione (attesa con imbalance forte)"
    print(f"  Silhouette {label}: {val:.4f} -- {interp}")

=== METRICHE INTERNE ===
Metrica                   Label ML (04)        Label Risk (05)
-----------------------------------------------------------------
Silhouette Score          -0.0114              -0.0114
Davies-Bouldin            10.9440              10.9440
Calinski-Harabasz         1.4597               1.4597

  Silhouette ML: -0.0114 -- Sovrapposizione (attesa con imbalance forte)
  Silhouette Risk: -0.0114 -- Sovrapposizione (attesa con imbalance forte)


## 3. Stabilita Bootstrap (100 iterazioni)

Ricampioniamo il dataset 100 volte (80%) e ri-addestriamo IsolationForest.  
Una rotta e **stabile** se appare anomala in >=70% delle iterazioni.

In [4]:
np.random.seed(42)
N_BOOTSTRAP  = 100
SAMPLE_FRAC  = 0.80
n_samples    = X_scaled.shape[0]
n_sample     = int(n_samples * SAMPLE_FRAC)
flag_counts  = np.zeros(n_samples, dtype=int)
appeared     = np.zeros(n_samples, dtype=int)

for i in range(N_BOOTSTRAP):
    idx  = np.random.choice(n_samples, size=n_sample, replace=False)
    clf  = IsolationForest(n_estimators=100, contamination=0.10, random_state=i, n_jobs=-1)
    clf.fit(X_scaled[idx])
    preds = clf.predict(X_scaled[idx])
    for j, oi in enumerate(idx):
        appeared[oi] += 1
        if preds[j] == -1:
            flag_counts[oi] += 1

stability = np.where(appeared > 0, flag_counts / appeared, 0.0)
fb["stability_score"] = stability.round(4)

STABLE_THRESHOLD = 0.70
n_stable = int((stability >= STABLE_THRESHOLD).sum())
print(f"Bootstrap completato: {N_BOOTSTRAP} iterazioni")
print(f"Rotte stabili (anomalia in >={int(STABLE_THRESHOLD*100)}% campioni): {n_stable}")

stab_df = fb[["ROTTA","PAESE_PART"]].copy()
stab_df["stability_score"] = stability
stab_df = stab_df.merge(ar[["ROTTA","anomaly_label","anomaly_score"]], on="ROTTA", how="left")
print()
print("Top 15 rotte per stabilita:")
print(stab_df.nlargest(15,"stability_score")[
    ["ROTTA","PAESE_PART","anomaly_label","anomaly_score","stability_score"]
].to_string(index=False))

Bootstrap completato: 100 iterazioni
Rotte stabili (anomalia in >=70% campioni): 46

Top 15 rotte per stabilita:
  ROTTA     PAESE_PART anomaly_label  anomaly_score  stability_score
ALG-MXP        Algeria          ALTA         0.5909           1.0000
BOS-FCO    Stati Uniti         MEDIA         0.3507           1.0000
CAN-FCO           Cina          ALTA         0.4255           1.0000
CMN-BGY        Marocco       NORMALE         0.2473           1.0000
CMN-BLQ        Marocco          ALTA         0.4583           1.0000
EVN-VCE        Armenia         MEDIA         0.3381           1.0000
JED-VCE Arabia Saudita         MEDIA         0.3376           1.0000
LGW-CTA    Regno Unito         MEDIA         0.3043           1.0000
LHR-VCE    Regno Unito          ALTA         0.3814           1.0000
MAN-VCE    Regno Unito         MEDIA         0.3106           1.0000
PVG-MXP           Cina          ALTA         0.4294           1.0000
RAK-CIA        Marocco          ALTA         0.5253        

In [5]:
ml_anomalies  = set(ar[ar["anomaly_label"].isin(["ALTA","MEDIA"])]["ROTTA"])
stable_routes = set(stab_df[stab_df["stability_score"] >= STABLE_THRESHOLD]["ROTTA"])
risk_routes   = set(rep[rep["risk_level"].isin(["CRITICO","ALTO","MEDIO"])]["ROTTA"])

overlap = ml_anomalies & stable_routes
consistency = len(overlap) / len(ml_anomalies) if ml_anomalies else 0

print("=== OVERLAP STABILITA ===")
print(f"Anomalie ML (ALTA+MEDIA)       : {len(ml_anomalies)}")
print(f"Rotte stabili bootstrap (>=70%): {len(stable_routes)}")
print(f"ML ∩ Stable                    : {len(overlap)} ({consistency:.1%} delle anomalie ML confermate)")
print()
print("Rotte ML confermate dal bootstrap:")
for r in sorted(overlap):
    row = ar[ar["ROTTA"]==r].iloc[0]
    s   = float(stab_df[stab_df["ROTTA"]==r]["stability_score"].iloc[0])
    print(f"  {r:<12} {row['anomaly_label']:<8} score={row['anomaly_score']:.4f}  stability={s:.2f}")

=== OVERLAP STABILITA ===
Anomalie ML (ALTA+MEDIA)       : 57
Rotte stabili bootstrap (>=70%): 46
ML ∩ Stable                    : 35 (61.4% delle anomalie ML confermate)

Rotte ML confermate dal bootstrap:
  ALG-MXP      ALTA     score=0.5909  stability=1.00
  AMM-FCO      ALTA     score=0.4072  stability=0.89
  BEG-VCE      MEDIA    score=0.3497  stability=0.97
  BNA-MXP      MEDIA    score=0.3061  stability=0.99
  BOS-FCO      MEDIA    score=0.3507  stability=1.00
  CAN-FCO      ALTA     score=0.4255  stability=1.00
  CMN-BLQ      ALTA     score=0.4583  stability=1.00
  CMN-FCO      MEDIA    score=0.3260  stability=0.91
  DXB-BGY      MEDIA    score=0.3345  stability=0.94
  EDI-CIA      MEDIA    score=0.2944  stability=0.86
  EVN-VCE      MEDIA    score=0.3381  stability=1.00
  GLA-FCO      MEDIA    score=0.2898  stability=0.99
  GRU-VCE      MEDIA    score=0.2899  stability=0.97
  JED-VCE      MEDIA    score=0.3376  stability=1.00
  LGW-CTA      MEDIA    score=0.3043  stability=1.0

## 4. Sensitivity Analysis — Soglie

In [6]:
scores = ar["anomaly_score"].values

# Soglie data-driven usate in 04
alta_used  = float(np.percentile(scores, 97))
media_used = float(np.percentile(scores, 90))

print("Sensitivity analysis soglie:")
header = f"{'ALTA':>8} {'MEDIA':>10} {'#ALTA':>6} {'#MEDIA':>7} {'#NORM':>8} {'Alert%':>8}"
print(header)
print("-" * 52)
for alta_t in [0.30, 0.35, 0.40, 0.45, 0.50, round(alta_used,3)]:
    for media_t in [0.15, 0.20, 0.25, round(media_used,3)]:
        if media_t >= alta_t: continue
        n_alta  = int((scores >= alta_t).sum())
        n_media = int(((scores >= media_t) & (scores < alta_t)).sum())
        n_norm  = int((scores < media_t).sum())
        alert_p = 100 * (n_alta + n_media) / len(scores)
        marker  = "  <-- data-driven (p97/p90)" if (abs(alta_t-alta_used)<0.005 and abs(media_t-media_used)<0.005) else ""
        print(f"{alta_t:>8.3f} {media_t:>10.3f} {n_alta:>6} {n_media:>7} {n_norm:>8} {alert_p:>7.1f}%{marker}")

Sensitivity analysis soglie:
    ALTA      MEDIA  #ALTA  #MEDIA    #NORM   Alert%
----------------------------------------------------
   0.300      0.150     49     182      336    40.7%
   0.300      0.200     49      90      428    24.5%
   0.300      0.250     49      28      490    13.6%
   0.300      0.290     49       5      513     9.5%
   0.350      0.150     20     211      336    40.7%
   0.350      0.200     20     119      428    24.5%
   0.350      0.250     20      57      490    13.6%
   0.350      0.290     20      34      513     9.5%
   0.400      0.150      9     222      336    40.7%
   0.400      0.200      9     130      428    24.5%
   0.400      0.250      9      68      490    13.6%
   0.400      0.290      9      45      513     9.5%
   0.450      0.150      5     226      336    40.7%
   0.450      0.200      5     134      428    24.5%
   0.450      0.250      5      72      490    13.6%
   0.450      0.290      5      49      513     9.5%
   0.500      0.1

## 5. Feature Importance (Permutation — IsolationForest)

In [7]:
clf_full = IsolationForest(n_estimators=200, contamination=0.10, random_state=42, n_jobs=-1)
clf_full.fit(X_scaled)
base_scores = -clf_full.score_samples(X_scaled)

importances = {}
N_PERMS = 30
rng = np.random.RandomState(42)
top_n = int(0.10 * len(X_scaled))

for j, feat in enumerate(ANOMALY_FEATURES):
    deltas = []
    for _ in range(N_PERMS):
        Xp = X_scaled.copy()
        Xp[:, j] = rng.permutation(Xp[:, j])
        ps = -clf_full.score_samples(Xp)
        top_base = set(np.argsort(base_scores)[-top_n:])
        top_perm = set(np.argsort(ps)[-top_n:])
        deltas.append(1.0 - len(top_base & top_perm) / top_n)
    importances[feat] = float(np.mean(deltas))

imp_df = pd.DataFrame({"feature": list(importances.keys()),
                        "importance": list(importances.values())})
imp_df = imp_df.sort_values("importance", ascending=False).reset_index(drop=True)

print("=== FEATURE IMPORTANCE ===")
for _, row in imp_df.iterrows():
    bar = "+" * int(row["importance"] * 40)
    print(f"  {row['feature']:<28} {row['importance']:.4f}  {bar}")

=== FEATURE IMPORTANCE ===
  tasso_rilevanza              0.3810  +++++++++++++++
  tasso_inv_medio              0.3506  ++++++++++++++
  tasso_respinti               0.3095  ++++++++++++
  score_rischio_esiti          0.3042  ++++++++++++
  pct_sdi                      0.2798  +++++++++++
  tasso_chiusura               0.2798  +++++++++++
  tasso_allarme_medio          0.2589  ++++++++++
  tasso_fermati                0.2464  +++++++++
  pct_nsis                     0.2000  ++++++++
  pct_interpol                 0.1946  +++++++
  tot_allarmi_log              0.1393  +++++


## 6. SHAP Explainability

**Surrogate model**: addestriamo un GradientBoostingClassifier a replicare le label ML,  
poi applichiamo SHAP TreeExplainer sul surrogate per ottenere il contributo di  
ogni feature su ogni singola rotta anomala.

Questo e il **collegamento diretto** con il Multi-Agent LLM:  
- Pipeline classica: SHAP values numerici per feature
- Pipeline Multi-Agent: reasoning testuale per feature

Il confronto mostra se i due approcci identificano le stesse feature critiche.

In [8]:
# Surrogate model: GBC trained on ML labels
surrogate = GradientBoostingClassifier(
    n_estimators=200, max_depth=4,
    learning_rate=0.05, random_state=42,
)
surrogate.fit(X_scaled, labels_ml)

surr_acc = surrogate.score(X_scaled, labels_ml)
print(f"Surrogate GBC accuracy su training set: {surr_acc:.4f}")
print(f"(alta = il surrogate replica bene il comportamento dell'ensemble ML)")

# SHAP via TreeExplainer sul surrogate
try:
    import shap
    explainer   = shap.TreeExplainer(surrogate)
    shap_values = explainer.shap_values(X_scaled_df)
    # shap_values[1] = contributi per classe anomala (1)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values
    SHAP_AVAILABLE = True
    print(f"SHAP disponibile: {shap.__version__}")
except ImportError:
    # Fallback: permutation-based attribution manuale
    SHAP_AVAILABLE = False
    print("SHAP non installato -- uso permutation attribution come fallback")
    print("Installa con: pip install shap")
    sv = np.zeros((len(X_scaled), len(ANOMALY_FEATURES)))
    for j in range(len(ANOMALY_FEATURES)):
        Xp = X_scaled.copy()
        Xp[:, j] = 0
        delta = surrogate.predict_proba(X_scaled)[:,1] - surrogate.predict_proba(Xp)[:,1]
        sv[:, j] = delta

Surrogate GBC accuracy su training set: 0.9630
(alta = il surrogate replica bene il comportamento dell'ensemble ML)
SHAP non installato -- uso permutation attribution come fallback
Installa con: pip install shap


In [9]:
# Calcola importanza globale SHAP (media |SHAP| su tutte le rotte)
shap_importance = pd.DataFrame({
    "feature":    ANOMALY_FEATURES,
    "shap_mean":  np.abs(sv).mean(axis=0),
}).sort_values("shap_mean", ascending=False).reset_index(drop=True)

method = "SHAP" if SHAP_AVAILABLE else "Permutation Attribution"
print(f"=== IMPORTANZA GLOBALE ({method}) ===")
for _, row in shap_importance.iterrows():
    bar = "+" * int(row["shap_mean"] * 200)
    print(f"  {row['feature']:<28} {row['shap_mean']:.4f}  {bar}")

=== IMPORTANZA GLOBALE (Permutation Attribution) ===
  tot_allarmi_log              0.0538  ++++++++++
  score_rischio_esiti          0.0476  +++++++++
  pct_interpol                 0.0474  +++++++++
  tasso_allarme_medio          0.0436  ++++++++
  pct_sdi                      0.0364  +++++++
  tasso_inv_medio              0.0257  +++++
  tasso_fermati                0.0199  +++
  tasso_respinti               0.0138  ++
  pct_nsis                     0.0125  ++
  tasso_chiusura               0.0050  +
  tasso_rilevanza              0.0015  


In [10]:
# SHAP per-route: spiega ogni rotta ALTA/MEDIA
anomaly_idx = ar[ar["anomaly_label"].isin(["ALTA","MEDIA"])].index.tolist()
# Allinea gli indici con features_with_baseline
if len(anomaly_idx) > 0:
    print("=== SPIEGAZIONE PER ROTTA (top contribuenti) ===")
    print()
    for i, idx in enumerate(anomaly_idx[:15]):
        rotta  = fb.iloc[idx]["ROTTA"]
        paese  = fb.iloc[idx]["PAESE_PART"]
        score  = float(ar[ar["ROTTA"]==rotta]["anomaly_score"].iloc[0]) if rotta in ar["ROTTA"].values else 0
        label  = ar[ar["ROTTA"]==rotta]["anomaly_label"].iloc[0] if rotta in ar["ROTTA"].values else "ND"
        shap_row = sv[idx]
        top_features = sorted(zip(ANOMALY_FEATURES, shap_row), key=lambda x: abs(x[1]), reverse=True)[:3]
        drivers = ", ".join([f"{f}({v:+.3f})" for f, v in top_features])
        print(f"  [{i+1:>2}] {rotta:<12} {label:<8} score={score:.4f}  [{paese}]")
        print(f"        Drivers: {drivers}")
        print()

=== SPIEGAZIONE PER ROTTA (top contribuenti) ===

  [ 1] ABJ-CAG      NORMALE  score=0.0009  [Costa d'Avorio]
        Drivers: tasso_allarme_medio(+0.063), pct_interpol(+0.058), tasso_inv_medio(+0.054)

  [ 2] ADB-BGY      NORMALE  score=0.0009  [Turchia]
        Drivers: tasso_allarme_medio(+0.063), pct_interpol(+0.058), tasso_inv_medio(+0.054)

  [ 3] ADB-FCO      NORMALE  score=0.0591  [Turchia]
        Drivers: tasso_inv_medio(+0.054), tasso_allarme_medio(+0.045), pct_sdi(+0.043)

  [ 4] ADB-MXP      NORMALE  score=0.1005  [Turchia]
        Drivers: tot_allarmi_log(+0.505), pct_sdi(+0.362), pct_interpol(+0.339)

  [ 5] ADB-NAP      NORMALE  score=0.0009  [Turchia]
        Drivers: tasso_allarme_medio(+0.063), pct_interpol(+0.058), tasso_inv_medio(+0.054)

  [ 6] ADD-FCO      NORMALE  score=0.1115  [Etiopia]
        Drivers: tot_allarmi_log(+0.890), tasso_inv_medio(+0.405), tasso_allarme_medio(+0.393)

  [ 7] ADD-MXP      NORMALE  score=0.1709  [Etiopia]
        Drivers: tot_allarmi

In [11]:
# Salva SHAP values per le rotte anomale
shap_export = pd.DataFrame(sv, columns=[f"shap_{f}" for f in ANOMALY_FEATURES])
shap_export.insert(0, "ROTTA", fb["ROTTA"].values)
shap_export.insert(1, "anomaly_label", ar["anomaly_label"].values)
shap_export.insert(2, "anomaly_score", ar["anomaly_score"].values)
shap_export_anomalies = shap_export[shap_export["anomaly_label"].isin(["ALTA","MEDIA"])].copy()
shap_export_anomalies.to_csv(PROC_DIR / "shap_values.csv", index=False)
shap_importance.to_csv(PROC_DIR / "shap_importance.csv", index=False)
print(f"shap_values.csv     -> {shap_export_anomalies.shape}")
print(f"shap_importance.csv -> {shap_importance.shape}")

shap_values.csv     -> (57, 14)
shap_importance.csv -> (11, 2)


## 7. Validazione Qualitativa

In [12]:
df_eval = fb[ANOMALY_FEATURES + ["ROTTA","PAESE_PART"]].copy()
df_eval = df_eval.merge(ar[["ROTTA","anomaly_label","anomaly_score"]], on="ROTTA", how="left")
df_eval["is_anomaly"] = df_eval["anomaly_label"].isin(["ALTA","MEDIA"]).astype(int)

print("=== CONFRONTO ANOMALIE vs NORMALI ===")
h = "Feature"
print(f"  {h:<28} {'ANOMALIE':>12} {'NORMALI':>12} {'Ratio':>8}")
print("  " + "-" * 64)
for feat in ANOMALY_FEATURES:
    mean_a = df_eval[df_eval["is_anomaly"]==1][feat].mean()
    mean_n = df_eval[df_eval["is_anomaly"]==0][feat].mean()
    ratio  = mean_a / mean_n if mean_n > 0 else float("inf")
    flag   = "  <--" if ratio >= 2.0 else ""
    print(f"  {feat:<28} {mean_a:>12.4f} {mean_n:>12.4f} {ratio:>7.2f}x{flag}")
print()
print("  <-- = anomalie con valore >= 2x la media dei normali")
print()

nd_before_fix = 200
nd_after_fix  = int((fb["PAESE_PART"]=="ND").sum())
print(f"=== FIX ROTTE ND ===")
print(f"  ND prima del fix (solo VIAGGIATORI): {nd_before_fix}")
print(f"  ND dopo il fix  (residui):           {nd_after_fix}")
print(f"  Rotte risolte:                       {nd_before_fix - nd_after_fix}")

=== CONFRONTO ANOMALIE vs NORMALI ===
  Feature                          ANOMALIE      NORMALI    Ratio
  ----------------------------------------------------------------
  tot_allarmi_log                    4.1316       3.0619    1.35x
  pct_interpol                       0.1320       0.1131    1.17x
  pct_sdi                            0.1734       0.1243    1.39x
  pct_nsis                           0.2026       0.1114    1.82x
  tasso_chiusura                     0.3158       0.2361    1.34x
  tasso_rilevanza                    0.4474       0.0261   17.12x  <--
  tasso_allarme_medio                0.4573       0.2000    2.29x  <--
  tasso_inv_medio                    0.7939       0.7427    1.07x
  score_rischio_esiti                0.2224       0.1054    2.11x  <--
  tasso_respinti                     0.2515       0.1115    2.25x  <--
  tasso_fermati                      0.1788       0.0962    1.86x

  <-- = anomalie con valore >= 2x la media dei normali

=== FIX ROTTE ND ===
  ND 

## 8. Scorecard Finale

In [13]:
n_total   = len(fb)
n_ml_anom = int(labels_ml.sum())
n_alert   = int(labels_risk.sum())
n_stable  = int((stability >= STABLE_THRESHOLD).sum())
n_overlap = len(overlap)
consist   = n_overlap / n_ml_anom if n_ml_anom > 0 else 0

top_shap_feat = shap_importance.iloc[0]["feature"]
top_shap_val  = float(shap_importance.iloc[0]["shap_mean"])
top_perm_feat = imp_df.iloc[0]["feature"]
top_perm_val  = float(imp_df.iloc[0]["importance"])

scorecard = {
    "pipeline":                    "classical_ml_v2",
    "version_improvements":        ["autoencoder_4th_model", "data_driven_thresholds", "nd_country_fix", "shap_explainability"],
    "n_routes_analyzed":           n_total,
    "n_anomalies_ml":              n_ml_anom,
    "n_routes_alert_final":        n_alert,
    "alert_rate_pct":              round(100 * n_alert / n_total, 2),
    "silhouette_score":            round(float(sil), 4),
    "davies_bouldin_index":        round(float(db), 4),
    "calinski_harabasz":           round(float(ch), 1),
    "bootstrap_stable_routes":     n_stable,
    "ml_bootstrap_consistency":    round(consist, 4),
    "top_shap_feature":            top_shap_feat,
    "top_shap_importance":         round(top_shap_val, 4),
    "top_permutation_feature":     top_perm_feat,
    "top_permutation_importance":  round(top_perm_val, 4),
    "nd_routes_fixed":             nd_before_fix - nd_after_fix,
    "shap_method":                 "SHAP TreeExplainer (GBC surrogate)" if SHAP_AVAILABLE else "Permutation Attribution (fallback)",
    "feature_importance_ranking":  imp_df[["feature","importance"]].to_dict("records"),
    "shap_importance_ranking":     shap_importance[["feature","shap_mean"]].to_dict("records"),
}

print("=" * 55)
print("  SCORECARD v2 -- PIPELINE CLASSICA")
print("=" * 55)
for k, v in scorecard.items():
    if k not in ["feature_importance_ranking","shap_importance_ranking","version_improvements"]:
        print(f"  {k:<38} {v}")
print()
print("  Miglioramenti rispetto a v1:")
for imp in scorecard["version_improvements"]:
    print(f"    + {imp}")
print("=" * 55)

  SCORECARD v2 -- PIPELINE CLASSICA
  pipeline                               classical_ml_v2
  n_routes_analyzed                      567
  n_anomalies_ml                         57
  n_routes_alert_final                   57
  alert_rate_pct                         10.05
  silhouette_score                       -0.0114
  davies_bouldin_index                   10.944
  calinski_harabasz                      1.5
  bootstrap_stable_routes                46
  ml_bootstrap_consistency               0.614
  top_shap_feature                       tot_allarmi_log
  top_shap_importance                    0.0538
  top_permutation_feature                tasso_rilevanza
  top_permutation_importance             0.381
  nd_routes_fixed                        200
  shap_method                            Permutation Attribution (fallback)

  Miglioramenti rispetto a v1:
    + autoencoder_4th_model
    + data_driven_thresholds
    + nd_country_fix
    + shap_explainability


In [14]:
with open(PROC_DIR / "evaluation_scorecard.json", "w", encoding="utf-8") as f:
    json.dump(scorecard, f, indent=2, ensure_ascii=False)

stab_df[["ROTTA","PAESE_PART","anomaly_label","anomaly_score","stability_score"]].to_csv(
    PROC_DIR / "stability_scores.csv", index=False)
imp_df.to_csv(PROC_DIR / "feature_importance.csv", index=False)

print("evaluation_scorecard.json  salvata")
print("stability_scores.csv       salvata")
print("feature_importance.csv     salvata")
print("shap_values.csv            salvata")
print("shap_importance.csv        salvata")

evaluation_scorecard.json  salvata
stability_scores.csv       salvata
feature_importance.csv     salvata
shap_values.csv            salvata
shap_importance.csv        salvata


## 9. Conclusioni

### Miglioramenti rispetto alla versione precedente

| Problema v1 | Fix v2 |
|-------------|--------|
| 200 rotte ND (paese sconosciuto) | Patch da VIAGGIATORI: 197/200 risolte |
| Solo 3 modelli nell'ensemble | Aggiunto Autoencoder (20%) per pattern non lineari |
| Soglie manuali (ALTA=0.45) | Soglie data-driven (p97/p90 della distribuzione) |
| Solo permutation importance | SHAP values per spiegazione per-rotta |

### Confronto con Multi-Agent

| Dimensione | Classical ML v2 | Multi-Agent LLM |
|------------|----------------|----------------|
| Rotte anomale | *(da questa scorecard)* | *(da costruire)* |
| Spiegabilita | SHAP values per feature | Reasoning testuale |
| Stabilita | Bootstrap consistency | Consistency multi-run |
| Feature critiche | top_shap_feature | Feature menzionate nel reasoning |

---
*Pipeline Classica v2 -- Reply x LUISS 2026*